<a href="https://colab.research.google.com/github/zhiyuguo-innis/MSSP607/blob/main/zhiyuguo-innis/MSSP607/Modules/Week_5/Week_05_Data_Wrangling_Join_Combine_Improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 05 — Data Wrangling: Join, Combine, and Reshape

This lesson introduces several of the most important pandas techniques for reorganizing and integrating data:

- hierarchical indexing (`MultiIndex`)
- database-style joins with `merge()`
- index-based joins with `join()`
- row/column concatenation with `concat()`
- filling overlapping data with `combine_first()`
- reshaping with `stack()`, `unstack()`, `pivot()`, and `melt()`

**Learning objectives**

By the end of the lesson, students should be able to:
1. Explain the difference between a key, an index, and a hierarchical index.
2. Select the correct join type (`inner`, `left`, `right`, `outer`) for a business question.
3. Detect and explain row multiplication caused by duplicate join keys.
4. Combine datasets vertically and horizontally with `pd.concat()`.
5. Reshape data between long and wide formats.
6. Validate a wrangling operation by checking row counts, missing values, and key uniqueness.

Content adapted from Wes McKinney, *Python for Data Analysis*, and the pandas documentation.


In [25]:
# Optional Google Colab setup.
# This cell runs only when the notebook is actually being executed in Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not running in Google Colab; skipping Google Drive mount.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# Optional course-specific connection/setup script.
# Keeping environment-specific configuration optional makes the lesson portable.
from pathlib import Path

connection_script = Path('/content/drive/MyDrive/google connection.py')
if IN_COLAB and connection_script.exists():
    get_ipython().run_line_magic('run', str(connection_script))
else:
    print('Course connection script not found or not required for this lesson.')


Course connection script not found or not required for this lesson.


In [27]:
# Set a course working directory only when it exists.
from pathlib import Path

course_dir = Path('/content/MSSP-607/WeeklyModules/Week05')
if course_dir.exists():
    %pushd /content/MSSP-607/WeeklyModules/Week05
else:
    print(f'Course directory not found: {course_dir}')
    print('Continuing from the current working directory.')


Course directory not found: /content/MSSP-607/WeeklyModules/Week05
Continuing from the current working directory.


In [28]:
# Core libraries used in this lesson.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.options.display.max_rows = 20
np.random.seed(12345)  # Reproducible demonstration data.
plt.rc('figure', figsize=(10, 6))
np.set_printoptions(precision=4, suppress=True)

print('pandas version:', pd.__version__)


pandas version: 2.2.3


## 1. Hierarchical Indexing (`MultiIndex`)

A standard pandas object has one label per axis position. A **hierarchical index** stores multiple index levels for each row or column. This is useful when one observation is naturally identified by more than one dimension—for example, *state + year* or *customer + order*.

**Teaching point:** A `MultiIndex` is not the same as having several ordinary columns. It changes how pandas labels and selects the data.


In [29]:
#Create two indexes on multiple levels: letters then numbers.
data = pd.Series(np.random.randn(9),
                 index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])
data

a  1   -0.204708
   2    0.478943
   3   -0.519439
b  1   -0.555730
   3    1.965781
c  1    1.393406
   2    0.092908
d  2    0.281746
   3    0.769023
dtype: float64

In [30]:
data.index  #List the tuples that form the indexes.

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

In [31]:
# Different ways to select from the first index level.
display(data['b'])
display(data['b':'c'])
display(data.loc[['b', 'd']])


,0
1,-0.555730
3,1.965781


b  1   -0.555730
   3    1.965781
c  1    1.393406
   2    0.092908
dtype: float64

b  1   -0.555730
   3    1.965781
d  2    0.281746
   3    0.769023
dtype: float64

In [32]:
# Select all rows whose second index level equals 2.
# The tuple syntax makes the two index dimensions explicit.
data.loc[(slice(None), 2)]


,0
a,0.478943
c,0.092908
d,0.281746


In [33]:
#Produce a Pandas Dataframe from the indexes.
data.unstack()

,1,2,3
a,-0.204708,0.478943,-0.519439
b,-0.555730,NaN,1.965781
c,1.393406,0.092908,NaN
d,NaN,0.281746,0.769023


In [34]:
#Unstack then restack the indexes.
data.unstack().stack()

a  1   -0.204708
   2    0.478943
   3   -0.519439
b  1   -0.555730
   3    1.965781
c  1    1.393406
   2    0.092908
d  2    0.281746
   3    0.769023
dtype: float64

In [35]:
#In NumPy, np.arange() is a function used to create an array with regularly
#spaced values within a specified interval. It returns an array containing
#numbers that start from a specified value and go up to, but not including,
#an endpoint value.
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                     columns=[['Ohio', 'Ohio', 'Colorado'],
                              ['Green', 'Red', 'Green']])
frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

In [36]:
#Rename indexes and columns of the dataframe for each index level.
frame.index.names = ['key1', 'key2']
frame.columns.names = ['state', 'color']
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [37]:
frame['Ohio']

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

A `MultiIndex` can also be built explicitly. The equivalent construction for the column labels above is:

```python
pd.MultiIndex.from_arrays(
    [['Ohio', 'Ohio', 'Colorado'], ['Green', 'Red', 'Green']],
    names=['state', 'color']
)
```

This explicit form is useful when index levels are being assembled programmatically.


### Reordering and Sorting Levels

In [38]:
frame.swaplevel('key1', 'key2')

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

In [39]:
# Reordering a MultiIndex and then sorting by a level.
display(frame.sort_index(level='key2'))
display(frame.swaplevel('key1', 'key2').sort_index(level='key2'))


state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

### Summary Statistics by Level

In [40]:
# Aggregation can be performed across columns or rows.
print('Row totals:')
display(frame.sum(axis=1))
print('Column totals:')
display(frame.sum(axis=0))


Row totals:


key1  key2
a     1        3
      2       12
b     1       21
      2       30
dtype: int64

Column totals:


state     color
Ohio      Green    18
          Red      22
Colorado  Green    26
dtype: int64

### Indexing with a DataFrame's columns

In [41]:
frame = pd.DataFrame({'a': range(7), 'b': range(7, 0, -1),
                      'c': ['one', 'one', 'one', 'two', 'two',
                            'two', 'two'],
                      'd': [0, 1, 2, 0, 1, 2, 3]})
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


In [42]:
frame2 = frame.set_index(['c', 'd'])
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

In [43]:
frame.set_index(['c', 'd'], drop=False)

a  b    c  d
c   d              
one 0  0  7  one  0
    1  1  6  one  1
    2  2  5  one  2
two 0  3  4  two  0
    1  4  3  two  1
    2  5  2  two  2
    3  6  1  two  3

In [44]:
frame2.reset_index()

,c,d,a,b
0,one,0,0,7
1,one,1,1,6
2,one,2,2,5
3,two,0,3,4
4,two,1,4,3
5,two,2,5,2
6,two,3,6,1


### Instructor note: choosing the right combining operation

Before demonstrating `merge()`, ask students: **“Are we matching records because they share a key, or are we simply stacking compatible data?”** This distinction separates `merge()` from `concat()` and prevents many early pandas mistakes.


## Combining and Merging Datasets

### Database-Style DataFrame Joins

`pd.merge()` combines rows by matching one or more key columns, similar to a SQL `JOIN`.

**Four questions to ask before every merge:**
1. Which column(s) form the key?
2. Are the keys unique in the left table, the right table, both, or neither?
3. Which rows should survive when a match is absent?
4. How will we validate the result?

A merge can silently create **more rows than either input** when duplicate keys exist. This is expected behavior for many-to-many joins, but it should never be a surprise.


In [45]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': range(7)})
df2 = pd.DataFrame({'key': ['a', 'b', 'd'],
                    'data2': range(3)})

print('Left table:')
display(df1)
print('Right table:')
display(df2)


Left table:


,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


Right table:


,key,data2
0,a,0
1,b,1
2,d,2


In [46]:
# If the common key column has the same name, pandas can infer it.
merged_default = pd.merge(df1, df2)
merged_default


,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


In [47]:
# Explicit is usually better for teaching and production code.
merged_on_key = pd.merge(df1, df2, on='key', validate='many_to_one')
merged_on_key


,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


In [48]:
df3 = pd.DataFrame({'lkey': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': range(7)})
df4 = pd.DataFrame({'rkey': ['a', 'b', 'd'],
                    'data2': range(3)})
pd.merge(df3, df4, left_on='lkey', right_on='rkey')

,lkey,data1,rkey,data2
0,b,0,b,1
1,b,1,b,1
2,a,2,a,0
3,a,4,a,0
4,a,5,a,0
5,b,6,b,1


In [49]:
# Outer join: preserve every key from both tables.
outer_merge = pd.merge(df1, df2, on='key', how='outer', indicator=True)
outer_merge


,key,data1,data2,_merge
0,a,2.0,0.0,both
1,a,4.0,0.0,both
2,a,5.0,0.0,both
3,b,0.0,1.0,both
4,b,1.0,1.0,both
5,b,6.0,1.0,both
6,c,3.0,NaN,left_only
7,d,NaN,2.0,right_only


In [50]:
# Demonstrate a many-to-many merge.
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'],
                    'data1': range(6)})
df2 = pd.DataFrame({'key': ['a', 'b', 'a', 'b', 'd'],
                    'data2': range(5)})

print('Left key counts:')
display(df1['key'].value_counts())
print('Right key counts:')
display(df2['key'].value_counts())

many_to_many = pd.merge(df1, df2, on='key', how='left', validate='many_to_many')
print(f'Rows before merge: left={len(df1)}, right={len(df2)}')
print(f'Rows after merge: {len(many_to_many)}')
many_to_many


Left key counts:


,count
key,
b,3
a,2
c,1


Right key counts:


,count
key,
a,2
b,2
d,1


Rows before merge: left=6, right=5
Rows after merge: 11


,key,data1,data2
0,b,0,1.0
1,b,0,3.0
2,b,1,1.0
3,b,1,3.0
4,a,2,0.0
5,a,2,2.0
6,c,3,NaN
7,a,4,0.0
8,a,4,2.0
9,b,5,1.0


In [51]:
# Inner join keeps only keys that appear in both tables.
inner_merge = pd.merge(df1, df2, on='key', how='inner')
inner_merge


,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,a,4,0
7,a,4,2
8,b,5,1
9,b,5,3


In [52]:
left = pd.DataFrame({'key1': ['foo', 'foo', 'bar'],
                     'key2': ['one', 'two', 'one'],
                     'lval': [1, 2, 3]})
right = pd.DataFrame({'key1': ['foo', 'foo', 'bar', 'bar'],
                      'key2': ['one', 'one', 'one', 'two'],
                      'rval': [4, 5, 6, 7]})
pd.merge(left, right, on=['key1', 'key2'], how='outer')

,key1,key2,lval,rval
0,bar,one,3.0,6.0
1,bar,two,NaN,7.0
2,foo,one,1.0,4.0
3,foo,one,1.0,5.0
4,foo,two,2.0,NaN


In [53]:
# If non-key column names overlap, suffixes keep their origins clear.
print('Merge on key1 only:')
display(pd.merge(left, right, on='key1'))

print('Same merge with explicit suffixes:')
display(pd.merge(left, right, on='key1', suffixes=('_left', '_right')))


Merge on key1 only:


,key1,key2_x,lval,key2_y,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


Same merge with explicit suffixes:


,key1,key2_left,lval,key2_right,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


### Instructor note: merge validation

Use `validate=` whenever you know the intended relationship:

- `one_to_one`
- `one_to_many`
- `many_to_one`
- `many_to_many`

This turns assumptions about key uniqueness into executable checks. For production data pipelines, that is much safer than visually inspecting the result.


### Merging on Index

In [54]:
left1 = pd.DataFrame({'key': ['a', 'b', 'a', 'a', 'b', 'c'],
                      'value': range(6)})
right1 = pd.DataFrame({'group_val': [3.5, 7]}, index=['a', 'b'])

display(left1)
display(right1)

pd.merge(left1, right1, left_on='key', right_index=True, validate='many_to_one')


,key,value
0,a,0
1,b,1
2,a,2
3,a,3
4,b,4
5,c,5


,group_val
a,3.5
b,7.0


,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0


In [55]:
pd.merge(left1, right1, left_on='key', right_index=True, how='outer')

,key,value,group_val
0,a,0,3.5
2,a,2,3.5
3,a,3,3.5
1,b,1,7.0
4,b,4,7.0
5,c,5,NaN


In [56]:
lefth = pd.DataFrame({'key1': ['Ohio', 'Ohio', 'Ohio', 'Nevada', 'Nevada'],
                      'key2': [2000, 2001, 2002, 2001, 2002],
                      'data': np.arange(5.)})
righth = pd.DataFrame(
    np.arange(12).reshape((6, 2)),
    index=pd.MultiIndex.from_arrays(
        [['Nevada', 'Nevada', 'Ohio', 'Ohio', 'Ohio', 'Ohio'],
         [2001, 2000, 2000, 2000, 2001, 2002]],
        names=['key1', 'key2']
    ),
    columns=['event1', 'event2']
)

display(lefth)
display(righth)


,key1,key2,data
0,Ohio,2000,0.0
1,Ohio,2001,1.0
2,Ohio,2002,2.0
3,Nevada,2001,3.0
4,Nevada,2002,4.0


event1  event2
key1   key2                
Nevada 2001       0       1
       2000       2       3
Ohio   2000       4       5
       2000       6       7
       2001       8       9
       2002      10      11

In [57]:
print('Inner merge on two left-side columns and a MultiIndex on the right:')
display(pd.merge(lefth, righth,
                 left_on=['key1', 'key2'], right_index=True))

print('Outer merge:')
display(pd.merge(lefth, righth,
                 left_on=['key1', 'key2'], right_index=True, how='outer'))


Inner merge on two left-side columns and a MultiIndex on the right:


,key1,key2,data,event1,event2
0,Ohio,2000,0.0,4,5
0,Ohio,2000,0.0,6,7
1,Ohio,2001,1.0,8,9
2,Ohio,2002,2.0,10,11
3,Nevada,2001,3.0,0,1


Outer merge:


,key1,key2,data,event1,event2
4,Nevada,2000,NaN,2.0,3.0
3,Nevada,2001,3.0,0.0,1.0
4,Nevada,2002,4.0,NaN,NaN
0,Ohio,2000,0.0,4.0,5.0
0,Ohio,2000,0.0,6.0,7.0
1,Ohio,2001,1.0,8.0,9.0
2,Ohio,2002,2.0,10.0,11.0


In [58]:
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]],
                     index=['a', 'c', 'e'],
                     columns=['Ohio', 'Nevada'])
right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                      index=['b', 'c', 'd', 'e'],
                      columns=['Missouri', 'Alabama'])

display(left2)
display(right2)
pd.merge(left2, right2, how='outer', left_index=True, right_index=True)


,Ohio,Nevada
a,1.0,2.0
c,3.0,4.0
e,5.0,6.0


,Missouri,Alabama
b,7.0,8.0
c,9.0,10.0
d,11.0,12.0
e,13.0,14.0


,Ohio,Nevada,Missouri,Alabama
a,1.0,2.0,NaN,NaN
b,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0
d,NaN,NaN,11.0,12.0
e,5.0,6.0,13.0,14.0


In [59]:
left2.join(right2, how='outer')

,Ohio,Nevada,Missouri,Alabama
a,1.0,2.0,NaN,NaN
b,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0
d,NaN,NaN,11.0,12.0
e,5.0,6.0,13.0,14.0


In [60]:
left1.join(right1, on='key')

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0
5,c,5,NaN


In [61]:
another = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [16., 17.]],
                       index=['a', 'c', 'e', 'f'],
                       columns=['New York', 'Oregon'])

display(another)
print('Default index join:')
display(left2.join([right2, another]))
print('Outer index join:')
display(left2.join([right2, another], how='outer'))


,New York,Oregon
a,7.0,8.0
c,9.0,10.0
e,11.0,12.0
f,16.0,17.0


Default index join:


,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1.0,2.0,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0,9.0,10.0
e,5.0,6.0,13.0,14.0,11.0,12.0


Outer index join:


,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1.0,2.0,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0,9.0,10.0
e,5.0,6.0,13.0,14.0,11.0,12.0
b,NaN,NaN,7.0,8.0,NaN,NaN
d,NaN,NaN,11.0,12.0,NaN,NaN
f,NaN,NaN,NaN,NaN,16.0,17.0


### Concatenating Along an Axis

Use `pd.concat()` when you want to **stack compatible objects**, not match rows by a relational key.

- `axis=0` stacks rows vertically.
- `axis=1` places objects side-by-side and aligns by index.
- `join='outer'` keeps the union of labels.
- `join='inner'` keeps only labels shared by all objects.
- `ignore_index=True` creates a fresh integer index.
- `keys=` adds a new hierarchical level that records each object's source.

**Rule of thumb:** `merge()` matches records by keys; `concat()` appends/aligned objects along an axis.


In [62]:
arr = np.arange(12).reshape((3, 4))
arr
np.concatenate([arr, arr], axis=1)

array([[ 0,  1,  2,  3,  0,  1,  2,  3],
       [ 4,  5,  6,  7,  4,  5,  6,  7],
       [ 8,  9, 10, 11,  8,  9, 10, 11]])

In [63]:
s1 = pd.Series([0, 1], index=['a', 'b'])
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'e'])
s3 = pd.Series([5, 6], index=['f', 'g'])

In [64]:
pd.concat([s1, s2, s3])

,0
a,0
b,1
c,2
d,3
e,4
f,5
g,6


In [65]:
pd.concat([s1, s2, s3], axis=1)

,0,1,2
a,0.0,NaN,NaN
b,1.0,NaN,NaN
c,NaN,2.0,NaN
d,NaN,3.0,NaN
e,NaN,4.0,NaN
f,NaN,NaN,5.0
g,NaN,NaN,6.0


In [66]:
s4 = pd.concat([s1, s3])
print('Combined Series s4:')
display(s4)

print('Outer alignment along columns:')
display(pd.concat([s1, s4], axis=1))

print('Inner alignment along columns:')
display(pd.concat([s1, s4], axis=1, join='inner'))


Combined Series s4:


,0
a,0
b,1
f,5
g,6


Outer alignment along columns:


,0,1
a,0.0,0
b,1.0,1
f,NaN,5
g,NaN,6


Inner alignment along columns:


,0,1
a,0,0
b,1,1


In [67]:
#Concatenate the two dataframes and reindex with new attributes.
pd.concat([s1, s4], axis=1).reindex(['a', 'c', 'b', 'e'])

,0,1
a,0.0,0.0
c,NaN,NaN
b,1.0,1.0
e,NaN,NaN


In [68]:
result = pd.concat([s1, s1, s3], keys=['one', 'two', 'three'])
result
result.unstack()

,a,b,f,g
one,0.0,1.0,NaN,NaN
two,0.0,1.0,NaN,NaN
three,NaN,NaN,5.0,6.0


In [69]:
pd.concat([s1, s2, s3], axis=1, keys=['one', 'two', 'three'])

,one,two,three
a,0.0,NaN,NaN
b,1.0,NaN,NaN
c,NaN,2.0,NaN
d,NaN,3.0,NaN
e,NaN,4.0,NaN
f,NaN,NaN,5.0
g,NaN,NaN,6.0


In [70]:
df1 = pd.DataFrame(np.arange(6).reshape(3, 2), index=['a', 'b', 'c'],
                   columns=['one', 'two'])
df2 = pd.DataFrame(5 + np.arange(4).reshape(2, 2), index=['a', 'c'],
                   columns=['three', 'four'])

display(df1)
display(df2)
pd.concat([df1, df2], axis=1, keys=['level1', 'level2'])


,one,two
a,0,1
b,2,3
c,4,5


,three,four
a,5,6
c,7,8


level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

In [71]:
pd.concat({'level1': df1, 'level2': df2}, axis=1)

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

In [72]:
pd.concat([df1, df2], axis=1, keys=['level1', 'level2'],
          names=['upper', 'lower'])

upper level1     level2     
lower    one two  three four
a          0   1    5.0  6.0
b          2   3    NaN  NaN
c          4   5    7.0  8.0

In [73]:
df1 = pd.DataFrame(np.random.randn(3, 4), columns=['a', 'b', 'c', 'd'])
df2 = pd.DataFrame(np.random.randn(2, 3), columns=['b', 'd', 'a'])

display(df1)
display(df2)


,a,b,c,d
0,1.246435,1.007189,-1.296221,0.274992
1,0.228913,1.352917,0.886429,-2.001637
2,-0.371843,1.669025,-0.438570,-0.539741


,b,d,a
0,0.476985,3.248944,-1.021228
1,-0.577087,0.124121,0.302614


In [74]:
pd.concat([df1, df2], ignore_index=True)

,a,b,c,d
0,1.246435,1.007189,-1.296221,0.274992
1,0.228913,1.352917,0.886429,-2.001637
2,-0.371843,1.669025,-0.438570,-0.539741
3,-1.021228,0.476985,NaN,3.248944
4,0.302614,-0.577087,NaN,0.124121


### Instructor checkpoint

Ask students to predict the result of concatenating two Series with different indexes along `axis=1`. The important idea is that pandas aligns by **labels**, not simply by row position.


### Combining Data with Overlap

Sometimes two objects describe the same observations but one has missing values where the other has data. In that case, we may want to fill gaps rather than create additional rows or columns.

`combine_first()` keeps the calling object's non-missing values and fills its missing values from the other object after index/column alignment.


In [75]:
a = pd.Series([np.nan, 2.5, np.nan, 3.5, 4.5, np.nan],
              index=['f', 'e', 'd', 'c', 'b', 'a'])
b = pd.Series(np.arange(len(a), dtype=np.float64),
              index=['f', 'e', 'd', 'c', 'b', 'a'])

# Label-based assignment is explicit and avoids ambiguous positional indexing.
b.loc['a'] = np.nan

display(a)
display(b)

# NumPy approach: choose b where a is missing; otherwise choose a.
pd.Series(np.where(a.isna(), b, a), index=a.index)


,0
f,NaN
e,2.5
d,NaN
c,3.5
b,4.5
a,NaN


,0
f,0.0
e,1.0
d,2.0
c,3.0
b,4.0
a,NaN


,0
f,0.0
e,2.5
d,2.0
c,3.5
b,4.5
a,NaN


In [76]:
b[:-2].combine_first(a[2:])

,0
a,NaN
b,4.5
c,3.0
d,2.0
e,1.0
f,0.0


In [77]:
df1 = pd.DataFrame({'a': [1., np.nan, 5., np.nan],
                    'b': [np.nan, 2., np.nan, 6.],
                    'c': range(2, 18, 4)})
df2 = pd.DataFrame({'a': [5., 4., np.nan, 3., 7.],
                    'b': [np.nan, 3., 4., 6., 8.]})

display(df1)
display(df2)
combined = df1.combine_first(df2)
combined


,a,b,c
0,1.0,NaN,2
1,NaN,2.0,6
2,5.0,NaN,10
3,NaN,6.0,14


,a,b
0,5.0,NaN
1,4.0,3.0
2,NaN,4.0
3,3.0,6.0
4,7.0,8.0


,a,b,c
0,1.0,NaN,2.0
1,4.0,2.0,6.0
2,5.0,4.0,10.0
3,3.0,6.0,14.0
4,7.0,8.0,NaN


## 3. Reshaping and Pivoting

Wrangling often changes the **shape** of data without changing its underlying information.

- **Wide format:** one row per entity, many measurement columns.
- **Long format:** one row per entity-variable observation.

Long format is often convenient for grouped analysis and visualization; wide format is often convenient for human-readable reports and some modeling workflows.


### Reshaping with Hierarchical Indexing

In [78]:
data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(['Ohio', 'Colorado'], name='state'),
                    columns=pd.Index(['one', 'two', 'three'],
                    name='number'))
data

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


In [79]:
result = data.stack()
result

state     number
Ohio      one       0
          two       1
          three     2
Colorado  one       3
          two       4
          three     5
dtype: int64

In [80]:
result.unstack()

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


In [81]:
result.unstack(0)
result.unstack('state')

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


In [82]:
s1 = pd.Series([0, 1, 2, 3], index=['a', 'b', 'c', 'd'])
s2 = pd.Series([4, 5, 6], index=['c', 'd', 'e'])
data2 = pd.concat([s1, s2], keys=['one', 'two'])
data2
data2.unstack()

,a,b,c,d,e
one,0.0,1.0,2.0,3.0,NaN
two,NaN,NaN,4.0,5.0,6.0


In [83]:
wide = data2.unstack()
print('Wide form:')
display(wide)

print('Stacked back to long form (missing combinations dropped by default):')
display(wide.stack())

print('Preserving missing combinations:')
try:
    display(wide.stack(future_stack=True))
except TypeError:
    display(wide.stack(dropna=False))


Wide form:


,a,b,c,d,e
one,0.0,1.0,2.0,3.0,NaN
two,NaN,NaN,4.0,5.0,6.0


Stacked back to long form (missing combinations dropped by default):


one  a    0.0
     b    1.0
     c    2.0
     d    3.0
two  c    4.0
     d    5.0
     e    6.0
dtype: float64

Preserving missing combinations:


one  a    0.0
     b    1.0
     c    2.0
     d    3.0
     e    NaN
two  a    NaN
     b    NaN
     c    4.0
     d    5.0
     e    6.0
dtype: float64

In [84]:
df = pd.DataFrame({'left': result, 'right': result + 5},
                  columns=pd.Index(['left', 'right'], name='side'))
df
df.unstack('state')

side   left          right         
state  Ohio Colorado  Ohio Colorado
number                             
one       0        3     5        8
two       1        4     6        9
three     2        5     7       10

In [85]:
df.unstack('state').stack('side')

/tmp/ipykernel_2121/3997798123.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.unstack('state').stack('side')


state         Ohio  Colorado
number side                 
one    left      0         3
       right     5         8
two    left      1         4
       right     6         9
three  left      2         5
       right     7        10

### Instructor note: long versus wide data

A useful mental model is:

- `melt()` makes a dataset **longer and narrower**.
- `pivot()` / `unstack()` usually make a dataset **wider and shorter**.

Ask students to identify the observational unit before and after each reshape.


### Pivoting “Long” to “Wide” Format

In [86]:
# Portable macroeconomic example.
# Prefer statsmodels' built-in dataset when available; otherwise use a small fallback dataset.
try:
    import statsmodels.api as sm
    data = sm.datasets.macrodata.load_pandas().data.copy()
except Exception:
    data = pd.DataFrame({
        'year': [2000, 2000, 2000, 2000, 2001, 2001, 2001, 2001],
        'quarter': [1, 2, 3, 4, 1, 2, 3, 4],
        'realgdp': [100, 101, 102, 103, 104, 105, 106, 107],
        'infl': [2.1, 2.2, 2.0, 2.3, 2.4, 2.2, 2.1, 2.0],
        'unemp': [4.0, 4.1, 4.2, 4.1, 4.0, 3.9, 3.8, 3.7]
    })

periods = pd.PeriodIndex(year=data['year'].astype(int),
                         quarter=data['quarter'].astype(int),
                         name='date')
columns = pd.Index(['realgdp', 'infl', 'unemp'], name='item')
data = data.reindex(columns=columns)
data.index = periods.to_timestamp(how='end').normalize()
ldata = data.stack().reset_index(name='value')


/tmp/ipykernel_2121/2021893804.py:15: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  periods = pd.PeriodIndex(year=data['year'].astype(int),


In [87]:
ldata[:10]

,date,item,value
0,1959-03-31,realgdp,2710.349
1,1959-03-31,infl,0.000
2,1959-03-31,unemp,5.800
3,1959-06-30,realgdp,2778.801
4,1959-06-30,infl,2.340
5,1959-06-30,unemp,5.100
6,1959-09-30,realgdp,2775.488
7,1959-09-30,infl,2.740
8,1959-09-30,unemp,5.300
9,1959-12-31,realgdp,2785.204


In [88]:
pivoted = ldata.pivot(index='date', columns='item', values='value')
pivoted

item,infl,realgdp,unemp
date,,,
1959-03-31,0.00,2710.349,5.8
1959-06-30,2.34,2778.801,5.1
1959-09-30,2.74,2775.488,5.3
1959-12-31,0.27,2785.204,5.6
1960-03-31,2.31,2847.699,5.2
...,...,...,...
2008-09-30,-3.16,13324.600,6.0
2008-12-31,-8.79,13141.920,6.9
2009-03-31,0.94,12925.410,8.1


In [89]:
ldata['value2'] = np.random.randn(len(ldata))
ldata[:10]

,date,item,value,value2
0,1959-03-31,realgdp,2710.349,0.523772
1,1959-03-31,infl,0.000,0.000940
2,1959-03-31,unemp,5.800,1.343810
3,1959-06-30,realgdp,2778.801,-0.713544
4,1959-06-30,infl,2.340,-0.831154
5,1959-06-30,unemp,5.100,-2.370232
6,1959-09-30,realgdp,2775.488,-1.860761
7,1959-09-30,infl,2.740,-0.860757
8,1959-09-30,unemp,5.300,0.560145
9,1959-12-31,realgdp,2785.204,-1.265934


In [90]:
# If we omit values=, pivot creates a hierarchical column index for every value column.
ldata['value2'] = np.random.randn(len(ldata))
pivoted_multi = ldata.pivot(index='date', columns='item')
display(pivoted_multi.head())

print('Selecting only the original value block:')
display(pivoted_multi['value'].head())


value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-03-31  0.00  2710.349   5.8 -0.247423  0.820211  0.302271
1959-06-30  2.34  2778.801   5.1 -0.942369  0.543980 -1.266383
1959-09-30  2.74  2775.488   5.3 -0.720102  0.937250 -1.593952
1959-12-31  0.27  2785.204   5.6 -0.958704 -0.375498  0.794336
1960-03-31  2.31  2847.699   5.2  0.543710 -1.605108  0.925166

Selecting only the original value block:


item,infl,realgdp,unemp
date,,,
1959-03-31,0.00,2710.349,5.8
1959-06-30,2.34,2778.801,5.1
1959-09-30,2.74,2775.488,5.3
1959-12-31,0.27,2785.204,5.6
1960-03-31,2.31,2847.699,5.2


In [91]:
unstacked = ldata.set_index(['date', 'item']).unstack('item')
unstacked[:7]

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-03-31  0.00  2710.349   5.8 -0.247423  0.820211  0.302271
1959-06-30  2.34  2778.801   5.1 -0.942369  0.543980 -1.266383
1959-09-30  2.74  2775.488   5.3 -0.720102  0.937250 -1.593952
1959-12-31  0.27  2785.204   5.6 -0.958704 -0.375498  0.794336
1960-03-31  2.31  2847.699   5.2  0.543710 -1.605108  0.925166
1960-06-30  0.14  2834.390   5.2 -0.399592 -1.469629  1.417343
1960-09-30  2.70  2839.022   5.6  1.844805 -0.897609  1.253168

### Pivoting “Wide” to “Long” Format

In [92]:
df = pd.DataFrame({'key': ['foo', 'bar', 'baz'],
                   'A': [1, 2, 3],
                   'B': [4, 5, 6],
                   'C': [7, 8, 9]})
df

,key,A,B,C
0,foo,1,4,7
1,bar,2,5,8
2,baz,3,6,9


In [93]:
melted = pd.melt(df, ['key'])
melted

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6
6,foo,C,7
7,bar,C,8
8,baz,C,9


In [94]:
reshaped = melted.pivot(index='key', columns='variable', values='value')
reshaped

variable,A,B,C
key,,,
bar,2,5,8
baz,3,6,9
foo,1,4,7


In [95]:
reshaped.reset_index()

variable,key,A,B,C
0,bar,2,5,8
1,baz,3,6,9
2,foo,1,4,7


In [96]:
pd.melt(df, id_vars=['key'], value_vars=['A', 'B'])

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6


In [97]:
print('Melt only selected numeric columns:')
display(pd.melt(df, value_vars=['A', 'B', 'C']))

print('Including key as a measured variable changes its role and usually is not what we want:')
display(pd.melt(df, value_vars=['key', 'A', 'B']))


Melt only selected numeric columns:


,variable,value
0,A,1
1,A,2
2,A,3
3,B,4
4,B,5
5,B,6
6,C,7
7,C,8
8,C,9


Including key as a measured variable changes its role and usually is not what we want:


,variable,value
0,key,foo
1,key,bar
2,key,baz
3,A,1
4,A,2
5,A,3
6,B,4
7,B,5
8,B,6


In [98]:
# Restore the previous directory only if %pushd was used successfully.
if course_dir.exists():
    %popd


## Conclusion and Validation Checklist

The key skill in data wrangling is not memorizing functions—it is choosing the operation that matches the structure of the problem.

| Goal | Main pandas tool |
|---|---|
| Match records by one or more keys | `pd.merge()` |
| Align objects by index | `.join()` |
| Stack objects by rows or columns | `pd.concat()` |
| Fill missing values from overlapping data | `.combine_first()` |
| Move index levels into columns | `.unstack()` |
| Move columns into index levels | `.stack()` |
| Convert tidy/long data to wide data | `.pivot()` |
| Convert wide data to long data | `pd.melt()` |

### Validate every wrangling step

After a merge, concat, or reshape, ask:

1. **Shape:** Did the number of rows and columns change as expected?
2. **Keys:** Are supposed-to-be-unique keys still unique?
3. **Missingness:** Did the operation create unexpected `NaN` values?
4. **Coverage:** Which records matched and which did not?
5. **Meaning:** Does one row still represent what you think it represents?

### Suggested in-class challenge

Create two small DataFrames: one with students and majors, and one with majors and department chairs. Merge them so every student remains in the result. Then use `indicator=True` to identify any student whose major did not match a department record.
